In [9]:
from SPARQLWrapper import SPARQLWrapper, JSON

# DATA RETRIEVAL FROM GRAPHDB
def _get_candidate_terms(repo, candidate_only=True):
    try:
        # specify the repository
        sparql = SPARQLWrapper(f'{repo}')

        # query => retrieving data
        query_all = '''
            PREFIX onner: <http://purl.org/spatialai/onner/onner-full#>
            PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

            SELECT ?paraId ?paragraph ?entId ?entity ?offset ?length ?label ?status ?annotator ?datetime
            WHERE {
                ?paraId onner:directlyContainsLabeledTerm ?entId .
                
                ?entId rdf:type onner:LabeledTerm ;
                    onner:labeledTermText ?entity ;
                    onner:offset ?offset ;
                    onner:length ?length ;
                    onner:hasLabeledTermStatus ?status .
                
                ?status onner:statusAssignmentDate ?datetime ;
                		onner:statusAssignedBy ?annotator ;
                		onner:hasLabeledTermLabel ?labelId .
                
                ?labelId rdf:type onner:Label ;
                		 onner:labelText ?label .
                
                ?paraId rdf:type onner:Paragraph ;
                		onner:paragraphText ?paragraph .
                
                {
                    SELECT DISTINCT ?paraId
                    WHERE {
                        ?entId2 rdf:type onner:LabeledTerm ;
                        	   onner:labeledTermDirectlyContainedBy ?paraId ;
                               onner:hasLabeledTermStatus ?status .
                        ?status rdf:type onner:CandidateStatus .   
            
                        {
                            SELECT ?entId2 (COUNT(?allStatus) AS ?statusCount)
                            WHERE {
                                ?entId2 onner:hasLabeledTermStatus ?allStatus .
                            }
                            GROUP BY ?entId2
                            HAVING (?statusCount = 1)
                        }
                    }
                }
            }
            ORDER BY ?paraId ?offset
        '''

        query_candidate_only = '''
            PREFIX onner: <http://purl.org/spatialai/onner/onner-full#>
            PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
            
            SELECT ?paraId ?paragraph ?entId ?entity ?offset ?length ?label ?status ?annotator ?datetime
            WHERE {
                ?entId rdf:type onner:LabeledTerm ;
                    onner:labeledTermText ?entity ;
                    onner:offset ?offset ;
                    onner:length ?length ;
                	onner:labeledTermDirectlyContainedBy ?paraId ;
                    onner:hasLabeledTermStatus ?status .
            
                ?status rdf:type onner:CandidateStatus ;
                		onner:statusAssignmentDate ?datetime ;
                		onner:statusAssignedBy ?annotator ;
                		onner:hasLabeledTermLabel ?labelId .
                
                ?labelId rdf:type onner:Label ;
                		 onner:labelText ?label .
                
                ?paraId rdf:type onner:Paragraph ;
                		onner:paragraphText ?paragraph .
            
                {
                    SELECT ?entId (COUNT(?allStatus) AS ?statusCount)
                    WHERE {
                        ?entId onner:hasLabeledTermStatus ?allStatus .
                    }
                    GROUP BY ?entId
                    HAVING (?statusCount = 1)
                }
            }
            ORDER BY ?paraId ?offset
        '''

        if candidate_only:
            sparql.setQuery(query_candidate_only)
        else:
            sparql.setQuery(query_all)

        # convert results to JSON
        sparql.setReturnFormat(JSON)

        # execute query
        results = sparql.query().convert()

        return results

    except Exception as e:
        print(f'Error querying the SPARQL endpoint: {e}')
        return None

In [10]:
def _merge_entity_statuses(data):
    for annotation in data['annotations']:
        entities = annotation[2]['entities']
        merged = {}

        for ent_id, start, end, statuses in entities:
            key = (ent_id, start, end)

            if key not in merged:
                merged[key] = [ent_id, start, end, []]

            merged[key][3].extend(statuses)

        annotation[2]['entities'] = list(merged.values())

    return data

In [11]:
def generate_candidate_data(repo, candidate_only=True):
    results = _get_candidate_terms(repo, candidate_only)
    records = results['results']['bindings']
    length = len(records)
    start_index = 0
    index_ranges = []
    
    for i in range(length-1):
        if records[i]['paragraph']['value'] != records[i+1]['paragraph']['value']:
            end_index = i + 1
            index_ranges.append([start_index, end_index])
            start_index = end_index
    
    index_ranges.append([start_index, len(records)])
    
    annotations = []
    
    for range_ in index_ranges:
        start_index = range_[0]
        end_index = range_[1]
        
        paragraph = records[start_index]['paragraph']['value']
        entities = []
        
        for j in records[start_index:end_index]:
            paragraph_id = j['paraId']['value']
            paragraph_id = paragraph_id.split('#')[1]
            
            entity_id = j['entId']['value']
            entity_id = entity_id.split('#')[1]
            
            entity = j['entity']['value']
            offset = j['offset']['value']
            length = j['length']['value']
            label = j['label']['value']
            
            status = j['status']['value']
            status = status.split('#')[1]
            status = status.split('_')[0]  # need to update after adding date with status, e.g., Candidate_260701021632
            
            annotator = j['annotator']['value']
            annotator = annotator.split('#')[1]
            
            timestamp = j['datetime']['value']
    
            start_span = int(offset)
            end_span = int(start_span) + int(length)
            entities.append([entity_id, start_span, end_span, [[status, label, timestamp, annotator]]])
    
        annotations.append([paragraph_id, paragraph, {'entities': entities}])
    
    data = {
        'classes': [
            {'id': 1, 'name': 'CHEM_ENT', 'color': 'red-11'},
            {'id': 2, 'name': 'MAT_ENT_STRUCT', 'color': 'blue-11'},
            {'id': 3, 'name': 'MAT_ENT_UNSTRUCT', 'color': 'light-green-11'},
            {'id': 4, 'name': 'PROPERTY', 'color': 'yellow-11'},
            {'id': 5, 'name': 'END_USE', 'color': 'purple-11'},
            {'id': 6, 'name': 'PROCESS', 'color': 'orange-11'},
            {'id': 7, 'name': 'EQUIPMENT', 'color': 'teal-11'},
            {'id': 8, 'name': 'MEASUREMENT', 'color': 'pink-11'},
            {'id': 9, 'name': 'ABBREVIATION', 'color': 'brown-11'}
        ],
        'annotations': annotations
    }
    
    return _merge_entity_statuses(data)


In [14]:
import json
from datetime import datetime

def main():
    
    graphdb_repo = 'http://dev:7200/repositories/Demon_FOIS' 
    data = generate_candidate_data(repo=graphdb_repo)
    timestamp = datetime.now().strftime('%y%m%d%H%M%S')
    
    with open(f'candidate_data_{timestamp}.json', 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

In [15]:
if __name__ == '__main__':
    main()